<a href="https://colab.research.google.com/github/carlottaforza03/Progetto-IntroDSePC-Gruppo-6/blob/branch_Carlotta/notebooks/Progetto_IntroDSePC_gruppo_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Progetto del corso - IntroDSePC - a.a. 2025/2026

#Membri del gruppo 6


Il gruppo 6 è composto da:


*   **Gaia** **Bellomo**: matricola 1226500 gaia.bellomo3@studio.unibo.it
*   **Carlotta** **Forza**: matricola 1230704 carlotta.forza@studio.unibo.it
*   **Federica** **La** **Braca**: matricola 1216267 federica.labraca@studio.unibo.it





#Materiale del progetto

...

#Preparazione dell'ambiente

In [2]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn import svm, tree
from sklearn.metrics import *

Importazione del repository da GitHub

In [3]:
!git clone https://github.com/carlottaforza03/Progetto-IntroDSePC-Gruppo-6

Cloning into 'Progetto-IntroDSePC-Gruppo-6'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 32 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 10.68 KiB | 5.34 MiB/s, done.
Resolving deltas: 100% (6/6), done.


#Fase 1: Analisi e comprensione del Dataset

Inserire il Dataset

In [4]:
shopping = pd.read_csv("sample_data/Online_Shoppers.csv", sep=",")
print(shopping)

       Administrative  Administrative_Duration  Informational  \
0                   0                      0.0              0   
1                   0                      0.0              0   
2                   0                      0.0              0   
3                   0                      0.0              0   
4                   0                      0.0              0   
...               ...                      ...            ...   
12325               3                    145.0              0   
12326               0                      0.0              0   
12327               0                      0.0              0   
12328               4                     75.0              0   
12329               0                      0.0              0   

       Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                         0.0               1                 0.000000   
1                         0.0               2                64.000000 

In [ ]:
shopping.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [ ]:
valori_mancanti = shopping.isnull().sum()
print("Conteggio dei valori mancanti per ogni colonna:\n", valori_mancanti)
#SE TUTTI I RISULTATI SONO ZERO ABBIAMO LA CERTEZZA CHE IL DATASET NON PRESENTA VALORI NULLI

Conteggio dei valori mancanti per ogni colonna:
 Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues                 0
SpecialDay                 0
Month                      0
OperatingSystems           0
Browser                    0
Region                     0
TrafficType                0
VisitorType                0
Weekend                    0
Revenue                    0
dtype: int64


# Fase 2: Descrizione e comprensione del Dataset

Il progetto si focalizza sullo studio analitico del comportamento degli utenti durante le sessioni di navigazione su una piattaforma di e-commerce, sfruttando le informazioni raccolte nel dataset degli acquirenti online.

Lo scopo fondamentale dell'analisi consiste nel prevedere l'intenzione d'acquisto dei visitatori, identificando se una determinata sessione si concluderà o meno con una transazione andata a buon fine.

Attraverso l'esplorazione di metriche web come il tasso di rimbalzo, il valore delle pagine e la durata della navigazione, il progetto mira a fornire un'interpretazione critica delle dinamiche digitali per aiutare a comprendere quali fattori spingano concretamente un utente alla conversione.

##Domande/ipotesi

###Domanda 1
Quale sorgente di traffico garantisce il tasso di conversione in acquisti più elevato e risulta quindi essere il canale di acquisizione più redditizio per la piattaforma web?


In [ ]:
conversione_traffico = shopping.groupby('TrafficType')['Revenue'].mean() * 100
print("Tasso di conversione per ogni sorgente di traffico:\n", conversione_traffico)

Tasso di conversione per ogni sorgente di traffico:
 TrafficType
1     10.689514
2     21.645796
3      8.771930
4     15.434986
5     21.538462
6     11.936937
7     30.000000
8     27.696793
9      9.523810
10    20.000000
11    19.028340
12     0.000000
13     5.826558
14    15.384615
15     0.000000
16    33.333333
17     0.000000
18     0.000000
19     5.882353
20    25.252525
Name: Revenue, dtype: float64


Valutando l'efficacia dei canali di acquisizione, emerge chiaramente che la sorgente di traffico categorizzata con il numero 16 risulta essere la più redditizia, garantendo un tasso di conversione del 33.33%.

### Domanda 2
Esiste un forte sbilanciamento tra gli utenti che completano l'acquisto e quelli che abbandonano il sito?

In [ ]:
sbilanciamento_target = shopping['Revenue'].value_counts(normalize=True) * 100
print("Percentuale di acquisti e non acquisti:\n", sbilanciamento_target)

Percentuale di acquisti e non acquisti:
 Revenue
False    84.525547
True     15.474453
Name: proportion, dtype: float64


Dall'analisi della variabile target emerge un forte sbilanciamento delle classi: la percentuale di sessioni che terminano con un acquisto è del 15.47%, mentre il restante 84.53% si conclude con l'abbandono del sito.

###Domanda 3
Come si distribuisce il traffico in base alla tipologia di utente, calcolando la proporzione esatta tra i visitatori di ritorno e i nuovi clienti?

In [ ]:
tipologia_visitatori = shopping['VisitorType'].value_counts(normalize=True) * 100
print("Distribuzione percentuale dei tipi di visitatore:\n", tipologia_visitatori)

Distribuzione percentuale dei tipi di visitatore:
 VisitorType
Returning_Visitor    85.571776
New_Visitor          13.738848
Other                 0.689376
Name: proportion, dtype: float64


Il traffico del sito è dominato dai visitatori di ritorno, che rappresentano il 85.57% del totale, seguiti dai nuovi clienti che coprono una quota del 13.74%.

###Domanda 4
Ci sono valori anomali nella durata media delle sessioni? (utenti che risultano connessi per una quantità di ore irrealistica)

In [ ]:
colonne_durata = ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration']
analisi_durata = shopping[colonne_durata].describe()
print(analisi_durata)

       Administrative_Duration  Informational_Duration  \
count             12330.000000            12330.000000   
mean                 80.818611               34.472398   
std                 176.779107              140.749294   
min                   0.000000                0.000000   
25%                   0.000000                0.000000   
50%                   7.500000                0.000000   
75%                  93.256250                0.000000   
max                3398.750000             2549.375000   

       ProductRelated_Duration  
count             12330.000000  
mean               1194.746220  
std                1913.669288  
min                   0.000000  
25%                 184.137500  
50%                 598.936905  
75%                1464.157214  
max               63973.522230  


La durata media di navigazione sulle pagine dei prodotti si attesta a 1194.75 secondi, ma il valore massimo registrato tocca un picco estremo di 63973.52 secondi, suggerendo la chiara presenza di valori anomali da trattare.

###Domanda 5
In quali mesi dell'anno si concentra il maggior numero di accessi alla piattaforma di e-commerce?

In [ ]:
traffico_mensile = shopping['Month'].value_counts()
print(traffico_mensile)

Month
May     3364
Nov     2998
Mar     1907
Dec     1727
Oct      549
Sep      448
Aug      433
Jul      432
June     288
Feb      184
Name: count, dtype: int64


Valutando la distribuzione temporale, il mese di May (maggio) risulta essere il periodo con la maggiore concentrazione di accessi, registrando un volume totale di 3364 sessioni.

###Domanda 6
Qual è il valore medio per il tasso di rimbalzo globale del sito? (calcolarlo e capire se la piattaforma presenta problemi tecnici o di usabilità).

In [ ]:
media_rimbalzo = shopping['BounceRates'].mean()
print("Rimbalzo medio totale:", media_rimbalzo)

Rimbalzo medio totale: 0.02219138047072182


Il tasso di rimbalzo medio globale pari al 2,22% ci da la possibilità di escludere in modo categorico gravi problematiche tecniche o strutturali e dimostra l'elevata usabilità del sito e confermando che gli utenti in entrata trovano un'interfaccia intuitiva ed in target con le loro intenzioni di ricerca.

###Domanda 7
Esistono colonne che esprimono concetti sovrapponibili o quasi identici, rischiando di inserire informazioni duplicate?

In [ ]:
correlazione_metriche = shopping[['BounceRates', 'ExitRates']].corr()
print(correlazione_metriche)

             BounceRates  ExitRates
BounceRates     1.000000   0.913004
ExitRates       0.913004   1.000000


L'indice di correlazione tra il tasso di rimbalzo e il tasso di uscita risulta essere di 0.91, indicando una fortissima somiglianza strutturale che rischia di inserire informazioni duplicate nel modello.

###Domanda 8
Quanto incidono numericamente le giornate speciali, verificando quante sessioni di navigazione avvengono in prossimità di eventi come San Valentino o Natale?

In [ ]:
volume_giornate_speciali = shopping[shopping['SpecialDay'] > 0]['SpecialDay'].count()
print("Totale sessioni vicine a festività:", volume_giornate_speciali)

Totale sessioni vicine a festività: 1251


## Statistiche descrittive

### Statistica 1
Frequenza degli acquisti nel dataset (variabile Revenue)

In [5]:
print(shopping['Revenue'].value_counts())
print(shopping['Revenue'].value_counts(normalize=True).mul(100).round(2))


Revenue
False    10422
True      1908
Name: count, dtype: int64
Revenue
False    84.53
True     15.47
Name: proportion, dtype: float64


Il dataset presenta un evidente sbilanciamento tra le classi, in quanto solo il 15,47% delle sessioni si conclude con un acquisto, mentre l'84,53% non genera conversioni.

###Statistica 2
Comportamento medio degli utenti nella consultazione dei prodotti

In [6]:
print(shopping['ProductRelated'].describe())


count    12330.000000
mean        31.731468
std         44.475503
min          0.000000
25%          7.000000
50%         18.000000
75%         38.000000
max        705.000000
Name: ProductRelated, dtype: float64


Gli utenti visitano in media 31,7 pagine prodotto per sessione, ma la mediana è pari a 18. La presenza di sessioni con valori molto elevati (fino a 705 pagine) suggerisce una distribuzione asimmetrica e la possibile presenza di outlier.

###Statistica 3
Incidenza delle sessioni prive di valore economico stimato (PageValues = 0)

In [7]:
zero_pv = (shopping['PageValues'] == 0).sum()
print(f"Sessioni con PageValues = 0: {zero_pv} ({zero_pv/len(shopping)*100:.1f}%)")
print(shopping.groupby(shopping['PageValues'] == 0)['Revenue'].mean().rename({True: 'PageValues = 0', False: 'PageValues > 0'}))


Sessioni con PageValues = 0: 9600 (77.9%)
PageValues
PageValues > 0    0.563370
PageValues = 0    0.038542
Name: Revenue, dtype: float64


Il 77,9% delle sessioni presenta un valore di PageValues pari a zero. Inoltre, il tasso di acquisto passa dal 3,85% per le sessioni con PageValues = 0 al 56,34% per quelle con PageValues > 0, indicando una forte relazione tra questa variabile e la conversione.

###Statistica 4
Relazione tra tasso di rimbalzo e tasso di uscita

In [8]:
shopping['Revenue_num'] = shopping['Revenue'].astype(int)
print(shopping[['BounceRates', 'ExitRates', 'Revenue_num']].corr())


             BounceRates  ExitRates  Revenue_num
BounceRates     1.000000   0.913004    -0.150673
ExitRates       0.913004   1.000000    -0.207071
Revenue_num    -0.150673  -0.207071     1.000000


Le variabili BounceRates ed ExitRates mostrano una correlazione molto elevata (0,913), indicando che descrivono comportamenti di navigazione fortemente simili.
Entrambe mostrano una correlazione negativa con Revenue, indicando che un maggiore tasso di abbandono del sito è associato a una minore probabilità di acquisto.

###Statistica 5
Distribuzione degli utenti tra visitatori nuovi e abituali

In [9]:
print(shopping['VisitorType'].value_counts(normalize=True).mul(100).round(2))
print(shopping.groupby('VisitorType')['Revenue'].mean().sort_values(ascending=False))


VisitorType
Returning_Visitor    85.57
New_Visitor          13.74
Other                 0.69
Name: proportion, dtype: float64
VisitorType
New_Visitor          0.249115
Other                0.188235
Returning_Visitor    0.139323
Name: Revenue, dtype: float64


Sebbene i Returning Visitor rappresentino la maggioranza delle sessioni (85,57%), il tasso di acquisto dei New Visitor (24,9%) risulta significativamente superiore a quello dei Returning Visitor (13,9%). Questo suggerisce che i nuovi utenti che raggiungono il sito potrebbero essere maggiormente orientati all'acquisto rispetto agli utenti abituali.

##Riflessioni critiche sui dati

###Riflessione 1
Comportamento d'acquisto per tipologia di utente


In [ ]:
contingenza_visitatori = pd.crosstab(shopping['VisitorType'], shopping['Revenue'], normalize='index') * 100
print("Percentuale di conversione (Revenue) in base al tipo di visitatore:")
print(contingenza_visitatori)


Percentuale di conversione (Revenue) in base al tipo di visitatore:
Revenue                False      True 
VisitorType                            
New_Visitor        75.088548  24.911452
Other              81.176471  18.823529
Returning_Visitor  86.067671  13.932329


Incrociando il tipo di visitatore (⁠VisitorType⁠) con il successo della sessione (⁠Revenue⁠), notiamo un dato interessante: un nuovo visitatore ha una probabilità di convertire (24,91%) quasi doppia, rispetto ad un visitatore che ritorna sul sito (13,93%).Questo dimostra che, mentre gli utenti fidelizzati esplorano il sito in modo informativo (generando molte sessioni ma senza acquistare sempre), i nuovi utenti approdano sulla piattaforma con un intento d'acquisto più immediato e mirato. In ottica di modellazione predittiva, la feature ⁠VisitorType⁠ agisce quindi come un forte discriminatore probabilistico.

###Riflessione 2
...

In [ ]:
# 1. Calcoliamo la media per i due gruppi
print("--- MEDIA PER GRUPPO ---")
print(shopping.groupby('Revenue')['ProductRelated_Duration'].mean())

# 2. Calcoliamo la mediana per i due gruppi
print("\n--- MEDIANA PER GRUPPO ---")
print(shopping.groupby('Revenue')['ProductRelated_Duration'].median())


--- MEDIA PER GRUPPO ---
Revenue
False    1069.987809
True     1876.209615
Name: ProductRelated_Duration, dtype: float64

--- MEDIANA PER GRUPPO ---
Revenue
False     510.19000
True     1109.90625
Name: ProductRelated_Duration, dtype: float64


Per chi compra e chi no la media è quasi il doppio della mediana. Quando la media viene trascinata così tanto verso l'alto rispetto al valore centrale, significa che la distribuzione dei dati ha una forte asimmetria a destra(positiva). La grande maggioranza degli utenti naviga sul sito per tempi "normali" ma ci sono alcuni utenti che naviga sul sito per tempi molto lunghi, questo fa in modo che il dataset presenti otulier strutturali che alterano la media campionaria facendola risultare molto elevata. La mediana risulta dunque l'indice più indicato per desvcrivere l'utente medio.

# Fase 3: Analisi esplorativa dei dati e visualizzazione

Possibili domande:
- I visitatori di ritorno (clienti abituali) hanno una probabilità matematicamente superiore di generare ricavi rispetto a chi visita il sito per la prima volta?

- Esiste una proporzionalità inversa diretta tra il tasso di uscita da una pagina e il completamento della transazione finale?

- Navigare durante i giorni di sabato e domenica porta ad un incremento effettivo della percentuale di conversione rispetto ai giorni feriali?

- Un punteggio alto nella colonna sulla metrica relativa al valore della pagina è il fattore più determinante per prevedere un acquisto?

#Fase 4: Modellazione

#Fase 5: Valutazione e interpretazione dei risultati

#Fase 6: Report scientifico in LaTeX (mettiamo qui il link?)